# World Cup Data Wrangling

In [ ]:
import pandas as pd

matches = pd.read_csv("../../data/raw/worldcup/matches.csv")
team_app = pd.read_csv("../../data/raw/worldcup/team_appearances.csv")
tournaments = pd.read_csv("../../data/raw/worldcup/tournaments.csv")
qualified = pd.read_csv("../../data/raw/worldcup/qualified_teams.csv")

In [3]:
# Year lookup from tournaments
year_map = tournaments.set_index("tournament_id")["year"].to_dict()
matches["year"] = matches["tournament_id"].map(year_map)

matches = matches[matches["year"] >= 2006].copy()
team_app = team_app[
    team_app["tournament_id"].isin(matches["tournament_id"].unique())
].copy()

In [4]:
# team_appearances has one row per team per match — pivot to wide
home = team_app[team_app["home_team"] == 1].copy()
away = team_app[team_app["away_team"] == 1].copy()

home = home[
    [
        "match_id",
        "tournament_id",
        "team_id",
        "team_name",
        "team_code",
        "goals_for",
        "goals_against",
        "goal_differential",
        "win",
        "draw",
        "lose",
        "extra_time",
        "penalty_shootout",
        "penalties_for",
        "penalties_against",
    ]
].add_suffix("_home")
away = away[
    [
        "match_id",
        "tournament_id",
        "team_id",
        "team_name",
        "team_code",
        "goals_for",
        "goals_against",
        "goal_differential",
        "win",
        "draw",
        "lose",
        "extra_time",
        "penalty_shootout",
        "penalties_for",
        "penalties_against",
    ]
].add_suffix("_away")

home = home.rename(
    columns={"match_id_home": "match_id", "tournament_id_home": "tournament_id"}
)
away = away.rename(
    columns={"match_id_away": "match_id", "tournament_id_away": "tournament_id"}
)

match_base = home.merge(away, on=["match_id", "tournament_id"])

In [5]:
meta_cols = [
    "match_id",
    "tournament_id",
    "year",
    "stage_name",
    "group_name",
    "group_stage",
    "knockout_stage",
    "match_date",
]
match_base = match_base.merge(matches[meta_cols], on=["match_id", "tournament_id"])

# Outcome from home team's perspective: W/D/L
match_base["result_home"] = match_base["win_home"].map({1: "W", 0: None})
match_base.loc[match_base["draw_home"] == 1, "result_home"] = "D"
match_base.loc[match_base["lose_home"] == 1, "result_home"] = "L"

match_base = match_base.sort_values(["year", "match_date"]).reset_index(drop=True)
print(match_base.shape)
print(
    match_base[
        [
            "year",
            "stage_name",
            "team_name_home",
            "team_name_away",
            "goals_for_home",
            "goals_for_away",
            "result_home",
        ]
    ].head(10)
)

(488, 35)
   year   stage_name         team_name_home  team_name_away  goals_for_home  \
0  2006  group stage                Germany      Costa Rica               4   
1  2006  group stage                 Poland         Ecuador               0   
2  2006  group stage                England        Paraguay               1   
3  2006  group stage    Trinidad and Tobago          Sweden               0   
4  2006  group stage              Argentina     Ivory Coast               2   
5  2006  group stage  Serbia and Montenegro     Netherlands               0   
6  2006  group stage                 Mexico            Iran               3   
7  2006  group stage                 Angola        Portugal               0   
8  2006  group stage              Australia           Japan               3   
9  2006  group stage          United States  Czech Republic               0   

   goals_for_away result_home  
0               2           W  
1               2           L  
2               0       

In [ ]:
# match_base.to_csv("../../data/raw/match_features_base.csv", index=False)